<a href="https://colab.research.google.com/github/VickkiMars/bible-seq/blob/main/BibleProjectTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 50.4 MB/s eta 0:00:00


# DATA

### **Cleaning**

In [3]:
!cp -r drive/MyDrive/Bible\ Project temp/

In [69]:
import fitz
import re

EFIK_PATH = "temp/Efik-All-Bible.pdf"
NIV_PATH = "temp/NIV-Bible.pdf"

In [70]:
import fitz  # PyMuPDF
import re

def get_all_verses(*pdf_paths):
    all_results = []
    verse_start_pattern = re.compile(r'^(\d+)\s*(.*)', re.MULTILINE)

    for path in pdf_paths:
        try:
            doc = fitz.open(path)
            file_verses = []
            current_verse = ""

            for page in doc:
                text = page.get_text()
                lines = text.splitlines()

                for line in lines:
                    line = line.strip()
                    if not line: continue

                    # Check if line starts with a verse number
                    match = verse_start_pattern.match(line)

                    if match:
                        # If we were already building a verse, save it before starting new one
                        if current_verse:
                            file_verses.append(current_verse)
                        current_verse = line
                    else:
                        # If it doesn't start with a digit, it's a continuation of the previous verse
                        if current_verse:
                            current_verse += " " + line

            # Append the very last verse of the file
            if current_verse:
                file_verses.append(current_verse)

            all_results.append(file_verses)
            doc.close()

        except Exception as e:
            print(f"Error loading {path}: {e}")
            all_results.append([]) # Keep the list index consistent even on failure

    return all_results



In [71]:
import re

def fix_verse_mismatches(file_verses):
    cleaned_file = []
    expected_verse = None
    current_chapter = None

    for verse in file_verses:
        match = re.match(r'^(\d+)', verse)

        # No leading number -> continuation of previous verse
        if not match:
            if cleaned_file:
                cleaned_file[-1] += " " + verse
            continue

        num = int(match.group(1))

        # 1. Initialization (First chapter anchor)
        if expected_verse is None:
            current_chapter = num
            expected_verse = 2  # The verse after a chapter anchor is always 2
            cleaned_file.append(verse)
            continue

        # 2. State Evaluation
        if num == expected_verse:
            # Normal verse progression (2, 3, 4...)
            expected_verse = num + 1
            cleaned_file.append(verse)

        elif current_chapter and num == current_chapter + 1:
            # New chapter anchor (e.g., transitioning from Chapter 6 to 7)
            current_chapter = num
            expected_verse = 2  # Reset expectation to verse 2
            cleaned_file.append(verse)

        else:
            # False positive (e.g., '45' from "45 feet high")
            # Fails both sequences, merge back into the previous valid verse
            if cleaned_file:
                cleaned_file[-1] += " " + verse

    return cleaned_file

In [86]:
efik, english = get_all_verses(EFIK_PATH, NIV_PATH)

In [87]:
efik = [ef for ef in efik if "Bible Society of Nigeria" not in ef][83:]
english = english[17:]

In [88]:
len(efik), len(english)

(31349, 31151)

In [93]:
english[0]

'1In the beginning God created the heavens and the earth.'

In [89]:
# english[141:141+22] Uncomment to see pre-fix state (verse 14-17)

In [90]:
efik, english = fix_verse_mismatches(efik), fix_verse_mismatches(english)

In [91]:
# english[141:141+32] Uncomment to see post-fix state (same)

In [92]:
len(efik), len(english)

(529, 857)

In [ ]:
english[-1]

In [77]:
def get_all_chapter_1s_efik(efik_list):
  chapter_1_indices = []
  for i, verse in enumerate(efik_list):
    # Check if the verse starts with '1' immediately followed by a letter, and has a length greater than 10 characters
    if re.match(r'^1[a-zA-Z]', verse) and len(verse) > 10:
      chapter_1_indices.append(i)
  return chapter_1_indices

In [78]:
efik_ids = get_all_chapter_1s_efik(efik)

In [79]:
efik_verse_gaps = [efik_ids[i+1] - efik_ids[i] for i in range(len(efik_ids)-1)]

In [80]:
import re

def get_leading_number(verse_string):
    """
    Extracts the integer value of leading digits from a verse string.
    Returns None if no leading digits are found.
    """
    match = re.match(r'^(\d+)', verse_string)
    if match:
        return int(match.group(1))
    return None

def get_all_chapter_1s_english(eng_list):
  chapter_1_indices = []
  for i in range(len(eng_list) - 1):
    current_verse = eng_list[i]
    next_verse = eng_list[i+1]

    try:
      upper_verse = eng_list[i+2]
    except IndexError as e:
      pass

    current_leading_number = get_leading_number(current_verse)
    next_leading_number = get_leading_number(next_verse)
    upper_leading_number = get_leading_number(upper_verse)

    # Chapter 1 is labelled 2
    if next_leading_number == 2 and upper_leading_number == 2 :
      chapter_1_indices.append(i+1)
      continue

    # Chapter 1 is labelled 1
    if next_leading_number == 2 and current_leading_number == 1:
      chapter_1_indices.append(i)

    # Chapter 1 is labelled any other number
    if next_leading_number == 2 and upper_leading_number > 2:
      chapter_1_indices.append(i)


  return chapter_1_indices

In [81]:
eng_ids = get_all_chapter_1s_english(english)

In [83]:
eng_ids

[0,
 0,
 31,
 31,
 56,
 80,
 106,
 138,
 160,
 184,
 206,
 235,
 267,
 299,
 319,
 337,
 361,
 382,
 409,
 452,
 526,
 591,
 681]

In [82]:
len(eng_ids)

23

In [56]:
def remove_duplicates_preserve_order(input_list):
    unique_list = []
    seen_elements = set()
    for item in input_list:
        if item not in seen_elements:
            unique_list.append(item)
            seen_elements.add(item)
    return unique_list

eng_ids = remove_duplicates_preserve_order(eng_ids)
print("Duplicates removed from eng_ids while preserving order.")
print(f"New length of eng_ids: {len(eng_ids)}")

Duplicates removed from eng_ids while preserving order.
New length of eng_ids: 1008


In [57]:
eng_ids = [eng_ids[i] for i in range(len(eng_ids)-1) if eng_ids[i] != eng_ids[i+1]]

In [58]:
english_verse_gaps = [eng_ids[i+1] - eng_ids[i] for i in range(len(eng_ids)-1)]

In [59]:
efik_verse_gaps[:15]

[30, 367, 182, 418, 1058, 696, 172, 302, 516, 376, 320, 536, 223, 336, 4378]

In [60]:
english_verse_gaps[:15]

[31, 21, 24, 26, 32, 22, 24, 21, 52, 32, 20, 34, 35, 26, 31]

'15This is how you are to build it: The ark is to be 450 feet long, 75 feet wide and45 feet high.'

In [21]:
eng_ids[5]

141

In [28]:
efik[138+32]

'11Ke ɔyɔhɔ isua uwem Noah ikie itiokiet, ke udiana akpa ɔfiɔŋ, ke ɔyɔhɔ usen ɔfiɔŋ efureba, ke usen oro kpukpru idim inyaŋ ibom asiaha, enyuŋ eberede window ikpa- enyɔŋ.'

In [38]:
new_eng = cleaner_pipeline(english[141:141+32])

In [44]:
len(new_eng)

28